In [1]:
!pip install -U bitsandbytes  ## Quantization

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.0 MB/s eta 0:00:00


In [2]:
!pip install trl ## Transformer reinforcement learning (SFT)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [3]:
import torch

torch.cuda.is_available()

True

In [4]:
import os
import torch
from datasets import load_dataset

In [5]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)

In [6]:
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

##Dataset

In [ ]:
# https://huggingface.co/datasets?sort=trending&search=llama

In [ ]:
System Message : Act as a AI Engineer
Human Message : Explain LoRA
AI Message : kjvjkbjbjb jhvjhv

In [ ]:
'''
<s>
[INST] <<SYS>>
Act as a AI Engineer
<</SYS>>
Explain LoRA
[/INST]
model answer ...
</s>
'''

In [7]:
# The instruction dataset to use
dataset_name = "mlabonne/guanaco-llama2-1k"

In [8]:
# Load dataset (you can process it here)
dataset = load_dataset(dataset_name, split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
dataset[10]

{'text': "<s>[INST] You will create a table with macronutrients, micronutrients and kcal of the following foods: 100g oatmeal, banana 100g, apple 100g, pasta ragù 100g [/INST] Here is a table showing the macronutrients, micronutrients, and kcal of 100g of oatmeal, banana, apple, and pasta ragù:\n\nFood\tCarbohydrates (g)\tFat (g)\tProtein (g)\tFiber (g)\tPotassium (mg)\tVitamin C (mg)\tkcal\nOatmeal\t66.3\t6.9\t16.9\t8.0\t-\t-\t389\nBanana\t23.0\t0.3\t1.3\t2.6\t358\t8.7\t89\nApple\t14.0\t0.2\t0.3\t2.4\t107\t4.6\t52\nPasta Ragù\t53.0\t12.0\t12.0\t2.0\t-\t-\t300\nNote: The above table is an estimation and may vary slightly depending on the specific </s><s>[INST] Thanks, that's awesome! [/INST] No prob, bud! </s>"}

In [ ]:
# The llama-2-chat model uses the following format to define system and instruction prompts:

'''

<s>
[INST] <<SYS>>
{{ system_prompt }}
<</SYS>>
{{ user_message }}
[/INST]
{{ model_answer }}
</s>


'''

'''
Let’s break down the different parts of the prompt structure:

<s>: the beginning of the entire sequence.
<<SYS>>: the beginning of the system message.
<</SYS>>: the end of the system message.
[INST]: the beginning of some instructions.
[/INST]: the end of some instructions.
{{ system_prompt }}: Where the user should edit the system prompt to give overall context to model responses.
{{ user_message }}: Where the user should provide instructions to the model for generating outputs.
{{ model_answer }} : Expected answer from LLM
'''

##BitsAndBytesConfig for quantization

In [10]:
# Load tokenizer and model with QLoRA configuration

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,

)

'''
BitsAndBytesConfig

>> load_in_4bit: we are loading the base model with a 4-bit quantization,
                  so we are setting this value to True.
'''

'\nBitsAndBytesConfig\n\n>> load_in_4bit: we are loading the base model with a 4-bit quantization,\n                  so we are setting this value to True.\n'

##Model & Tokenizer

In [11]:
# The model that you want to train from the Hugging Face hub
model_name = "NousResearch/Llama-2-7b-chat-hf"

In [12]:
# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

In [13]:
# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

##LoRA Config

In [ ]:
## Working of LoRA :
1. Create a copy of weight matrix (dw)
2. Freeze the original weights (W)
3. Decompose dw into low dimensions : A & B

## FineTune :
Fine tune (update) decomposed weight matrix (A & B)

In [ ]:
500*500
# 500*2, 2*500

In [14]:
# Load LoRA configuration
peft_config = LoraConfig(
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
'''
The LoraConfig has the following attributes.

>> lora_dropout: dropout probability of the LoRA layers.
          This parameter is used to avoid overfitting.
           This technique basically drop-outs some of the neurons during both
           forward and backward propagation, this will help in removing dependency
           on a single unit of neurons. We are setting this to 0.1 (which is 10%),
           which means each neuron has a dropout chance of 10%.
>> r: This is the dimension of the low-rank matrix,
      In this case, we are setting this to 64
       (which effectively means we will have 512x64 and 64x512 parameters in our LoRA adapter.
>> bias: We will not be training the bias in this example, so we are setting that to “none”.
        If we have to train the biases, we can set this to “all”, or if we want to train
        only the LORA biases then we can use “lora_only”
>> task_type: Since we are using the Causal language model, the task type we set to CAUSAL_LM.
'''

In [15]:
from peft import get_peft_model

In [16]:
lora_model = get_peft_model(model, peft_config)

##Training Arguments

In [17]:
# Set training parameters
training_arguments = TrainingArguments(
    output_dir = "/content/LoRA",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    save_steps=0,
    learning_rate=2e-4,
    weight_decay=0.001,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    report_to="none"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
'''
>> output_dir: The output directory is where the model predictions and checkpoints will be stored.
>> num_train_epochs: One training epoch.
>> per_device_train_batch_size: Batch size per GPU for training.
>> gradient_accumulation_steps: This refers to the number of steps required to accumulate the gradients during the update process.
>> Optim : Model optimizer (AdamW optimizer).
>> save_steps: How often to save a checkpoint of the model during training.
>> learning_rate: Initial learning rate.
>> weight_decay: Regularization technique applied to model weights to prevent overfitting.
>> max_grad_norm: Gradient clipping.
>> max_steps: Number of training steps.
>> warmup_ratio: Ratio of steps for a linear warmup.
>> group_by_length: This can significantly improve performance and accelerate the training process.
                    When set to `True`, samples of similar lengths are grouped together in
                    batches, making training more efficient. This avoids wasting computation
                    on padding tokens when sequences of different lengths are processed together.
'''

##Supervised Fine-Tuning(SFT)

In [ ]:
'''
Supervised Fine-Tuning(SFT) :
The TRL library from Hugging FAce provides an easy-to-use API to create SFT models
and train them on datasets.
'''

In [18]:
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_arguments,

)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss


In [ ]:
## Save finetuned model :
trainer.save_model("/content/LoRA")

##Merging Base Model & FineTuned Weights :

In [ ]:
# Reload model  and merge it with LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
)

model = PeftModel.from_pretrained(base_model, "/content/LoRA")

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name)


##Difference between LoRA and QLoRA

In [ ]:
# LoRA :

# Benefits :
  -- reduces memory requirements
  -- Faster finetuning
  -- Performance : LoRA maintain almost similae performance compared to traditional finetuning

In [ ]:
Q-LoRA :
 -- works same as LoRA just with quantization


In [ ]:
## How to choose :

Memory : QLoRA > LoRA
Performance : LoRA > QLoRA

###Pushing model to huggigface hub :

In [ ]:
## Create access token (Write)


In [ ]:
!huggingface-cli login

In [ ]:
model.push_to_hub("DattaUgale/<name-of-model>")

In [ ]:
DattaUgale --> username

<name-of-model>  --> name of model